# Local Jupyter Setup for Learning Spark V2

Utilities for converting Databricks paths to local paths.

In [1]:
import sys
from io import StringIO

old_stderr = sys.stderr
sys.stderr = StringIO()

from pyspark.sql import SparkSession
import os

# Create Spark session
spark = SparkSession.builder \
    .appName("LearningSparkV2") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready")

sys.stderr = old_stderr


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 05:11:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.1.1 ready


## Path Mappings

Replace Databricks paths with local equivalents:

In [2]:
# Get repo root
repo_root = "/Users/abhikashyap10/spark/LearningSparkV2"

# Databricks → Local mapping
DATABRICKS_PATHS = {
    "/databricks-datasets/learning-spark-v2/": f"{repo_root}/databricks-datasets/learning-spark-v2/",
    "/databricks-datasets/flights/": f"{repo_root}/databricks-datasets/learning-spark-v2/flights/",
    "/databricks-datasets/loans/": f"{repo_root}/databricks-datasets/learning-spark-v2/loans/",
    "/databricks-datasets/cctvVideos/": f"{repo_root}/databricks-datasets/learning-spark-v2/cctvVideos/",
}

def convert_path(databricks_path):
    """Convert Databricks dataset path to local path."""
    for db_prefix, local_prefix in DATABRICKS_PATHS.items():
        if databricks_path.startswith(db_prefix):
            local_path = databricks_path.replace(db_prefix, local_prefix)
            if os.path.exists(local_path):
                return local_path
    return databricks_path  # Return as-is if no mapping found

# Test
test_path = "/databricks-datasets/learning-spark-v2/SPARK_README.md"
converted = convert_path(test_path)
print(f"Databricks: {test_path}")
print(f"Local:      {converted}")
print(f"Exists:     {os.path.exists(converted)}")

Databricks: /databricks-datasets/learning-spark-v2/SPARK_README.md
Local:      /Users/abhikashyap10/spark/LearningSparkV2/databricks-datasets/learning-spark-v2/SPARK_README.md
Exists:     True


## Usage in Notebooks

Replace this:
```python
df = spark.read.text("/databricks-datasets/learning-spark-v2/SPARK_README.md")
```

With this:
```python
local_path = convert_path("/databricks-datasets/learning-spark-v2/SPARK_README.md")
df = spark.read.text(local_path)
```

## Databricks API Replacements

| Databricks | Local Replacement |
|-----------|------------------|
| `display(df)` | `df.show(truncate=False)` |
| `dbutils.fs.ls(path)` | `os.listdir(path)` |
| `display(dbutils.fs.head(...))` | `open(path).read()` |
| `/databricks-datasets/...` | See `convert_path()` above |
| SQL `%sql` cells | Convert to Python `spark.sql()` |
| `%md` markdown cells | Keep as-is |
